# Finetuning on REAL data

# 1. Load dataloaders 

In [ ]:
from data.dataloader import create_dataloaders

train_dataloader, val_dataloader = create_dataloaders(
    csv_path= "../../project_datasets/audio/tubular/parkinsons_real.csv",
    batch_size= 32,
    train_val_split= 0.5,
)

# 2. Load models

In [ ]:
from Models.efficientnetB1 import EfficientNet1D

model = EfficientNet1D()
model_name = "Audio_Tabular_Model"

# 3. Train models

In [ ]:
from training.trainer import train


train(
    model= model,
    train_dataloader=  train_dataloader,
    val_dataloader=  val_dataloader,
    
    model_name= model_name,
    run_name= model_name,
    
    # load_pretrained="checkpoints/best.pth",
    
    epochs= 50
)

In [ ]:
!tensorboard --logdir=runs

# 4. Plot confusion matrix (of the best model)

In [ ]:
import torch
from Models.efficientnetB1 import EfficientNet1D

model = EfficientNet1D()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
load_pretrained = "checkpoints/best.pth"

# load checkpoint
checkpoint = torch.load(load_pretrained, map_location=device)
# load model
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded pretrained model:")
print(f"- val_loss={checkpoint['val_loss']:.4f}")
print(f"- val_acc={checkpoint['val_acc']:.4f}")
print(f"- val_recall={checkpoint['val_recall']:.4f}")
print(f"- val_precision={checkpoint['val_precision']:.4f}")
print(f"- val_f1={checkpoint['val_f1']:.4f}")

In [ ]:
from training.confusion_mat import plot_confusion_matrix

plot_confusion_matrix(
    model=model,
    dataloader=val_dataloader,
    device=device,
    class_names=["Healthy", "PD"],
    # threshold=0.4,
)